# Aggregation

In [1]:
import os
import pandas as pd

In [2]:
# Base directory is the current directory containing Analysis.ipynb
base_dir = "."

# Collect renamed DataFrames
renamed_dfs = []

# Loop through folders in ResultsAnalysis/
for arch_folder in os.listdir(base_dir):
    arch_path = os.path.join(base_dir, arch_folder)

    if not os.path.isdir(arch_path):
        continue  # Skip files (like Analysis.ipynb itself)

    # Check for architecture folders (MGT, PIMGT_*)
    for seed_folder in os.listdir(arch_path):
        seed_path = os.path.join(arch_path, seed_folder)
        metrics_file = os.path.join(seed_path, "metrics.csv")

        if not os.path.isfile(metrics_file):
            continue  # Skip if metrics.csv is missing

        # Extract seed number (E25 -> 25)
        seed = seed_folder.replace("E", "")

        # Read the CSV
        df_metrics = pd.read_csv(metrics_file)

        # Rename columns: {architecture}_seed{seed}_{metric}
        new_columns = {
            col: f"{arch_folder}_seed{seed}_{col}" for col in df_metrics.columns
        }
        df_metrics.rename(columns=new_columns, inplace=True)

        renamed_dfs.append(df_metrics)

# Concatenate all into one DataFrame (aligned by prediction step index)
df_merged = pd.concat(renamed_dfs, axis=1)

# Preview
df_merged.head(10)

df_merged.to_csv("Aggregation.csv", index=False)

In [3]:
def aggregate_txt_files(filename, output_csv):
    all_dfs = []

    # Walk through architecture folders
    for arch_folder in os.listdir(base_dir):
        arch_path = os.path.join(base_dir, arch_folder)
        if not os.path.isdir(arch_path):
            continue

        # Walk through seed folders like E25, E42, etc.
        for seed_folder in os.listdir(arch_path):
            if not seed_folder.startswith("E"):
                continue

            seed_path = os.path.join(arch_path, seed_folder)
            file_path = os.path.join(seed_path, filename)

            if os.path.isfile(file_path):
                seed = seed_folder.replace("E", "")
                col_name = f"{arch_folder}_seed{seed}"

                # Read as a single-column DataFrame
                df = pd.read_csv(file_path, header=None, names=[col_name])
                all_dfs.append(df)

    # Combine all DataFrames by column (aligned by index)
    df_combined = pd.concat(all_dfs, axis=1)
    df_combined.to_csv(output_csv, index=False)
    return df_combined


In [4]:
def aggregate_val_maes(filename="val_maes.txt", output_csv="val_maes.csv"):
    all_dfs = []

    for arch_folder in os.listdir(base_dir):
        arch_path = os.path.join(base_dir, arch_folder)
        if not os.path.isdir(arch_path):
            continue

        for seed_folder in os.listdir(arch_path):
            if not seed_folder.startswith("E"):
                continue

            seed_path = os.path.join(arch_path, seed_folder)
            file_path = os.path.join(seed_path, filename)

            if os.path.isfile(file_path):
                seed = seed_folder.replace("E", "")
                col_name = f"{arch_folder}_seed{seed}"

                # Read and discard first column
                df = pd.read_csv(file_path, delim_whitespace=True, header=None, usecols=[1], names=[col_name])
                all_dfs.append(df)

    df_combined = pd.concat(all_dfs, axis=1)
    df_combined.to_csv(output_csv, index=False)
    return df_combined


In [5]:
# Convert and aggregate ave_losses.txt
df_losses = aggregate_txt_files("ave_losses.txt", "ave_losses.csv")

# Convert and aggregate val_maes.txt
df_val_maes = aggregate_val_maes()